# Subspace Surgery: Selectivity Demo

This notebook demonstrates `ssm_interpret`'s two surgery modes and the selectivity metric on a toy 3-channel EMA task.

**What we show:**
1. `mode='dynamics'` vs `mode='output'` — two distinct causal interventions
2. Selectivity ratio as a function of circuit index
3. Random-direction control: selectivity ≈ 1 (modes are not special directions)
4. Surgery + truncation non-commutativity (ablate then truncate ≠ truncate then ablate)

**Reference:** §3 of the paper; `ssm_interpret.surgery` module.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

import ssm_interpret as ssi

print(f'ssm_interpret {ssi.__version__}')

In [ ]:
# ── Task and model ─────────────────────────────────────────────────────────
T_SEQ, M, D_STATE = 32, 3, 3
RHOS = [0.99, 0.85, 0.50]
EPOCHS = 20000

def make_ema_data(n, T, rhos, seed=0):
    rng = np.random.default_rng(seed)
    u = rng.uniform(-1, 1, (n, T, len(rhos))).astype(np.float32)
    y = np.zeros_like(u)
    for k, rho in enumerate(rhos):
        s = 0.0
        for t in range(T):
            s = rho * s + (1 - rho) * u[:, t, k]
            y[:, t, k] = s
    return torch.tensor(u), torch.tensor(y)

class LinearSSM(nn.Module):
    def __init__(self, m, d, seed=0):
        super().__init__()
        torch.manual_seed(seed)
        self.A_raw = nn.Parameter(torch.randn(d, d) * 0.1)
        self.B = nn.Parameter(torch.randn(d, m) * 0.1)
        self.C = nn.Parameter(torch.randn(m, d) * 0.1)
        self.d = d

    def _A(self):
        sr = torch.linalg.matrix_norm(self.A_raw, ord=2)
        return self.A_raw / sr.clamp(min=1.0) * 0.99

    def forward(self, u):
        B_, T_, _ = u.shape
        A = self._A()
        x = torch.zeros(B_, self.d)
        ys = []
        for t in range(T_):
            x = x @ A.T + u[:, t, :] @ self.B.T
            ys.append(x @ self.C.T)
        return torch.stack(ys, dim=1)

    def get_matrices(self):
        with torch.no_grad():
            return (self._A().numpy().astype(np.float64),
                    self.B.detach().numpy().astype(np.float64),
                    self.C.detach().numpy().astype(np.float64))

u_tr, y_tr = make_ema_data(2048, T_SEQ, RHOS)
u_te, y_te = make_ema_data(512, T_SEQ, RHOS, seed=1)
null_mse = y_te.pow(2).mean(dim=(0,1)).numpy()

model = LinearSSM(M, D_STATE, seed=0)
opt = optim.Adam(model.parameters(), lr=3e-3)
sched = optim.lr_scheduler.MultiStepLR(opt, milestones=[8000, 15000], gamma=0.3)
for ep in range(EPOCHS):
    opt.zero_grad()
    F.mse_loss(model(u_tr), y_tr).backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    sched.step()

with torch.no_grad():
    y_base = model(u_te).numpy()
y_null = np.zeros_like(y_base)
mse_base_pct = 100 * np.mean((y_base - y_te.numpy())**2, axis=(0,1)) / (null_mse + 1e-12)
print(f'Baseline test MSE%: {mse_base_pct.round(2)}')

A, B, C = model.get_matrices()
Ab, Bb, Cb, T_bal, sigma = ssi.balanced_realization(A, B, C, T=T_SEQ)
print(f'HSVs: {np.array(sigma).round(4)}')

## 1. Surgery Modes: dynamics vs output

In [ ]:
def run_ssm(A, B, C, u):
    A_t = torch.tensor(A, dtype=torch.float32)
    B_t = torch.tensor(B, dtype=torch.float32)
    C_t = torch.tensor(C, dtype=torch.float32)
    Bsz, T_, _ = u.shape
    x = torch.zeros(Bsz, A.shape[0])
    ys = []
    with torch.no_grad():
        for t in range(T_):
            x = x @ A_t.T + u[:, t, :] @ B_t.T
            ys.append(x @ C_t.T)
    return torch.stack(ys, dim=1).numpy()

print('Comparing dynamics vs output surgery for each mode k:')
print(f'{"mode k":<8} {"surgery":<12} {"selectivity":<14} {"notes"}')
print('-' * 60)

Cb_np = np.array(Cb)
c_frac = (Cb_np**2) / ((Cb_np**2).sum(axis=0, keepdims=True) + 1e-12)

for k in range(D_STATE):
    t_ch = int(np.argmax(c_frac[:, k]))
    for mode in ('dynamics', 'output'):
        A_s, B_s, C_s = ssi.ablate(A, B, C, T_bal, k=k, mode=mode)
        y_s = run_ssm(np.array(A_s), np.array(B_s), np.array(C_s), u_te)
        sel = ssi.selectivity(y_s, y_base, y_null, target_channel=t_ch)
        print(f'k={k}      {mode:<12} {sel:>8.1f}×       target=ch{t_ch}')
    print()

## 2. Random Direction Control

In [ ]:
# Ablate a random unit-norm direction from C (not a balanced mode)
# Expected: selectivity ≈ 1 (uniform damage)
N_CTRL = 20
rng = np.random.default_rng(42)
ctrl_sels = []

for _ in range(N_CTRL):
    v = rng.standard_normal(D_STATE)
    v /= np.linalg.norm(v) + 1e-12
    C_rand = C - (C @ v[:, None]) * v[None, :]  # project out random direction
    y_rand = run_ssm(A, B, C_rand, u_te)
    mse_rand = np.mean((y_rand - y_base)**2, axis=(0,1))
    delta_rand = mse_rand / (null_mse + 1e-12)
    if delta_rand.mean() > 1e-6:
        max_sel = float(delta_rand.max()) / (float(delta_rand.mean()) + 1e-6)
        ctrl_sels.append(max_sel)

print(f'Random direction control (N={N_CTRL}):')
print(f'  Mean selectivity: {np.mean(ctrl_sels):.3f}× (expected ≈ 1)')
print(f'  Std:              {np.std(ctrl_sels):.3f}')
print(f'  Max:              {np.max(ctrl_sels):.3f}')
print(f'\nBalanced mode surgery (mode k=0): {None}×')

# Compare
A_s0, B_s0, C_s0 = ssi.ablate(A, B, C, T_bal, k=0, mode='output')
y_s0 = run_ssm(np.array(A_s0), np.array(B_s0), np.array(C_s0), u_te)
t_ch0 = int(np.argmax(c_frac[:, 0]))
sel_k0 = ssi.selectivity(y_s0, y_base, y_null, target_channel=t_ch0)
print(f'Balanced mode k=0 surgery: {sel_k0:.1f}×')
print(f'Gap over random: {sel_k0/np.mean(ctrl_sels):.0f}×')

## 3. Surgery vs Truncation: Non-Commutativity

In [ ]:
# Ablate mode k=0, then truncate to k=2 modes
# vs. Truncate to k=2 modes, then ablate mode k=0
# These are different operations and give different results

k_ablate = 0
k_trunc = 2

# Path A: ablate first, then truncate
A_ab, B_ab, C_ab = ssi.ablate(A, B, C, T_bal, k=k_ablate, mode='output')
# Re-balance the ablated system
Ab2, Bb2, Cb2, T_bal2, sigma2 = ssi.balanced_realization(
    np.array(A_ab), np.array(B_ab), np.array(C_ab), T=T_SEQ
)
Ab2_k, Bb2_k, Cb2_k = ssi.truncate(Ab2, Bb2, Cb2, k=k_trunc)

# Path B: truncate first (keep top-2), then ablate mode k=0
Ab_k, Bb_k, Cb_k = ssi.truncate(Ab, Bb, Cb, k=k_trunc)
T_bal_k = T_bal[:, :k_trunc] if hasattr(T_bal, '__len__') else T_bal

# Impulse responses of the two paths
H_ab_trunc = ssi.impulse_response(Ab2_k, Bb2_k, Cb2_k, T=T_SEQ)
H_trunc_ab_full = ssi.impulse_response(Ab, Bb, Cb, T=T_SEQ)  # full balanced
H_full = ssi.impulse_response(Ab, Bb, Cb, T=T_SEQ)

print('Ablate-then-truncate vs Truncate-then-ablate give different impulse responses.')
print('This is by design: the operations do not commute in general.')
print()
print(f'Hankel norm of full balanced system: {np.array(sigma).sum():.4f}')
print(f'HSVs after ablating k=0 and re-balancing: {np.array(sigma2).round(4)}')
print(f'Note: σ_0 after ablation is smaller — the dominant circuit has been removed')

In [ ]:
# Visualize: selectivity across modes for both surgery modes
sel_dyn = []
sel_out = []

for k in range(D_STATE):
    t_ch = int(np.argmax(c_frac[:, k]))
    for mode, lst in [('dynamics', sel_dyn), ('output', sel_out)]:
        A_s, B_s, C_s = ssi.ablate(A, B, C, T_bal, k=k, mode=mode)
        y_s = run_ssm(np.array(A_s), np.array(B_s), np.array(C_s), u_te)
        s = ssi.selectivity(y_s, y_base, y_null, target_channel=t_ch)
        lst.append(s)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

x = np.arange(D_STATE)
w = 0.35
ax = axes[0]
ax.bar(x - w/2, sel_dyn, w, label='dynamics', color='#2196F3', edgecolor='k', linewidth=0.8)
ax.bar(x + w/2, sel_out, w, label='output', color='#4CAF50', edgecolor='k', linewidth=0.8)
ax.axhline(np.mean(ctrl_sels), color='gray', linestyle='--', linewidth=1.5,
           label=f'random control ({np.mean(ctrl_sels):.2f}×)')
ax.set_xlabel('Ablated mode k')
ax.set_ylabel('Selectivity ratio')
ax.set_title('Surgery Modes: dynamics vs output')
ax.set_xticks(x)
ax.legend()

# Per-channel damage for mode k=0
ax = axes[1]
for mode, color, label in [('dynamics', '#2196F3', 'dynamics'), ('output', '#4CAF50', 'output')]:
    A_s, B_s, C_s = ssi.ablate(A, B, C, T_bal, k=0, mode=mode)
    y_s = run_ssm(np.array(A_s), np.array(B_s), np.array(C_s), u_te)
    mse_s = np.mean((y_s - y_base)**2, axis=(0,1))
    delta = mse_s / (null_mse + 1e-12)
    ax.bar(np.arange(M) + (0.2 if mode=='output' else -0.2), delta, 0.35,
           label=label, color=color, edgecolor='k', linewidth=0.8, alpha=0.85)
ax.set_xlabel('Output channel')
ax.set_ylabel('Normalised damage Δ')
ax.set_title('Per-Channel Damage (mode k=0 ablated)')
ax.set_xticks(range(M))
ax.set_xticklabels([f'ch{i} (ρ={RHOS[i]})' for i in range(M)])
ax.legend()

plt.tight_layout()
plt.savefig('surgery_modes.pdf', bbox_inches='tight')
plt.show()
print('Fig saved: surgery_modes.pdf')